# 2. The heavy top benchmark

This is the benchmark from the thesis (table 1): a fast-spinning top with a fixed point, $m = 15$ kg and $\omega_0 = (0, 150, -4.61538)$ rad/s. The disk is only its visual shape, so we pass the benchmark's mass and inertia explicitly.

In [ ]:
import sys
if sys.platform == "emscripten":  # the online version installs its packages here
    %pip install -q lib_rigid_body pythreejs

from netgen.occ import Box, Cylinder, Pnt
from lib_rigid_body import World
import numpy as np

In [ ]:
def heavy_top():
    world = World()
    disk = Cylinder(Pnt(0, .92, 0), (0, 1, 0), r=.55, h=.16)
    top = world.add(
        disk, at=(0, 1, 0), mass=15, inertia=np.diag([.234375, .46875, .234375]),
        spin=(0, 150, -4.61538), velocity=(4.61538, 0, 0),
    )
    world.pin(top, (0, 0, 0))
    return world, top

world, top = heavy_top()
recording = world.run(duration=1, h=0.001)
print(recording.summary())
recording.show(speed=0.25)  # quarter speed

## Second-order convergence

We compare the centre of mass at $t = 1$ with a reference computed by a different method (RK4 on Euler's equations with 80,000 steps). Halving $h$ should divide the error by 4.

In [ ]:
reference = np.array([0.173343964098, 0.640088592073, -0.748490791132])

errors = {}
for h in [1/250, 1/500, 1/1000]:
    world, top = heavy_top()
    errors[h] = np.linalg.norm(world.run(duration=1, h=h).positions[-1, 0] - reference)

previous = None
for h, error in errors.items():
    rate = "" if previous is None else f"rate {np.log2(previous / error):.2f}"
    print(f"h = {h:.5f}   error = {error:.3e}   {rate}")
    previous = error